In [ ]:
# === Notebook common preamble (loading llm_math package) ===
import sys
from pathlib import Path

# Candidate paths where llm_math package might be found
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Add parent directories of current directory to candidates (when running from notebooks/ folder)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Attempt to import llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Warning] Failed to load llm_math package: {e}")
    print("  Please clone the GitHub repo and run colab_setup.sh.")
# === preamble end ===

# Ch 12. Tokenizer — Turning Text into Numbers ⭐

> **Learning Objectives**
> - Understand why subword tokenization is the standard for LLMs
> - Implement the BPE algorithm from scratch in Python
> - Explain the differences between WordPiece and Unigram Language Model
> - Appreciate the advantages of byte-level tokenizers (Byte-level BPE)

## 12.1 Challenges of Tokenization

Three extremes:
- **Word-level**: "Hello, world!" → ["Hello", ",", "world", "!"]
  - Problem: Infinite vocabulary, unable to handle typos/neologisms, OOV (Out-Of-Vocabulary)
- **Character-level**: "Hello" → ['H', 'e', 'l', 'l', 'o']
  - Problem: Sequences too long, loss of meaningful units
- **Subword-level**: "unhappiness" → ["un", "happiness"] or ["un", "happ", "iness"]
  - **LLM Standard**: High-frequency tokens kept whole, low-frequency ones split

GPT-2: Byte-level BPE. BERT: WordPiece. T5: Unigram.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict

# Word-level tokenization example
text = "The quick brown fox jumps over the lazy dog."
word_tokens = text.lower().replace('.', ' .').split()
print(f"word level: {word_tokens}")

# Character-level
char_tokens = list(text.lower())
print(f"Character-level: {char_tokens}")

# Subword (conceptual example)
subword_tokens = ['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'over', 'lazy', 'dog', '.']
print(f"Subword level (example): {subword_tokens}")

## 12.2 Byte-Pair Encoding (BPE) — GPT-2’s Choice

**Algorithm**:
1. Initialize each word as a character sequence
2. Merge the most frequent adjacent pair (2-gram)
3. Repeat until vocabulary size is reached

Example: 'low' (5 times), 'lower' (2 times), 'newest' (6 times), 'widest' (3 times)

In the frequency table, the pair 'e'+'s' occurs 9 times → merge → 'es'

In [ ]:
# BPE direct implementation
from collections import Counter

class BPE:
    def __init__(self, num_merges=100):
        self.num_merges = num_merges
        self.merges = []  # (a, b) in order for merge rules
        self.vocab = set()

    def _get_word_freqs(self, corpus):
        """Word frequency computation."""
        word_freqs = Counter()
        for text in corpus:
            words = text.lower().split()
            for w in words:
                word_freqs[w] += 1
        return word_freqs

    def _get_pair_freqs(self, splits, word_freqs):
        """Adjacent pair frequency computation."""
        pair_freqs = Counter()
        for word, split in splits.items():
            for i in range(len(split) - 1):
                pair = (split[i], split[i+1])
                pair_freqs[pair] += word_freqs[word]
        return pair_freqs

    def _merge(self, pair, splits):
        """Merge pair in all words."""
        new_splits = {}
        a, b = pair
        for word, split in splits.items():
            new_split = []
            i = 0
            while i < len(split):
                if i < len(split) - 1 and split[i] == a and split[i+1] == b:
                    new_split.append(a + b)
                    i += 2
                else:
                    new_split.append(split[i])
                    i += 1
            new_splits[word] = new_split
        return new_splits

    def train(self, corpus):
        word_freqs = self._get_word_freqs(corpus)
        # Decompose each word into character-level (with </w> at end)
        splits = {w: list(w) + ['</w>'] for w in word_freqs}
        # Initial vocabulary
        self.vocab = set(c for w in word_freqs for c in w) | {'</w>'}

        for _ in range(self.num_merges):
            pair_freqs = self._get_pair_freqs(splits, word_freqs)
            if not pair_freqs:
                break
            best_pair = max(pair_freqs, key=pair_freqs.get)
            splits = self._merge(best_pair, splits)
            self.merges.append(best_pair)
            self.vocab.add(best_pair[0] + best_pair[1])

    def encode(self, word):
        """Encode word using trained rules."""
        split = list(word) + ['</w>']
        for a, b in self.merges:
            i = 0
            new_split = []
            while i < len(split):
                if i < len(split) - 1 and split[i] == a and split[i+1] == b:
                    new_split.append(a + b)
                    i += 2
                else:
                    new_split.append(split[i])
                    i += 1
            split = new_split
        return split

# Small corpus for training
corpus = [
    "low low low low low",
    "lower lower newest newest newest",
    "widest widest newest newest",
]
bpe = BPE(num_merges=10)
bpe.train(corpus)

print("Trained merge rules:")
for i, (a, b) in enumerate(bpe.merges, 1):
    print(f"  {i}. {a} + {b} -> {a+b}")
print(f"\nVocabulary Size: {len(bpe.vocab)}")

# Encoding test
for word in ['low', 'lowest', 'newer', 'widest']:
    tokens = bpe.encode(word)
    print(f"  {word:>10} -> {tokens}")

## 12.3 WordPiece — BERT's Choice

Similar to BPE but **with a different merging criterion**:

$$\text{score}(a, b) = \frac{\text{freq}(ab)}{\text{freq}(a) \cdot \text{freq}(b)}$$

BPE maximizes raw frequency, whereas WordPiece maximizes normalized frequency.  
This preference favors pairs that genuinely co-occur over those that appear together merely by chance.

In [ ]:
# Simple WordPiece implementation
class WordPiece:
    def __init__(self, num_merges=100):
        self.num_merges = num_merges
        self.merges = []

    def _get_pair_scores(self, splits, word_freqs):
        pair_freqs = Counter()
        char_freqs = Counter()
        for word, split in splits.items():
            for i in range(len(split) - 1):
                pair = (split[i], split[i+1])
                pair_freqs[pair] += word_freqs[word]
            for s in split:
                char_freqs[s] += word_freqs[word]
        scores = {p: f / (char_freqs[p[0]] * char_freqs[p[1]]) for p, f in pair_freqs.items()}
        return scores

    def _merge(self, pair, splits):
        new_splits = {}
        a, b = pair
        for word, split in splits.items():
            new_split = []
            i = 0
            while i < len(split):
                if i < len(split) - 1 and split[i] == a and split[i+1] == b:
                    new_split.append(a + b)
                    i += 2
                else:
                    new_split.append(split[i])
                    i += 1
            new_splits[word] = new_split
        return new_splits

    def train(self, corpus):
        word_freqs = Counter()
        for text in corpus:
            for w in text.lower().split():
                word_freqs[w] += 1
        # WordPiece uses ## prefix to mark internal subword tokens
        splits = {}
        for w in word_freqs:
            chars = list(w)
            splits[w] = [chars[0]] + ['##' + c for c in chars[1:]]
        for _ in range(self.num_merges):
            scores = self._get_pair_scores(splits, word_freqs)
            if not scores:
                break
            best = max(scores, key=scores.get)
            splits = self._merge(best, splits)
            self.merges.append(best)

wp = WordPiece(num_merges=10)
wp.train(corpus)
print("WordPiece Trained merges:")
for i, (a, b) in enumerate(wp.merges, 1):
    print(f"  {i}. {a} + {b} -> {a+b}")

## 12.4 Unigram Language Model — T5's Approach

BPE/WordPiece is **bottom-up** (merging from characters). Unigram is **top-down** (deleting from a large vocabulary).

1. Initial vocabulary = all subword candidates (large)  
2. Each subword probability $P(t) = \text{count}(t) / \sum \text{count}$  
3. Loss $\mathcal{L} = -\sum_i \log P(\mathbf{x}_i) = -\sum_i \log \sum_{\text{seg}} \prod P(t)$  
4. Delete the token whose removal causes the smallest increase in loss  
5. Repeat  

Optimal segmentation is found using the EM algorithm.

In [ ]:
# Simple unigram implementation (conceptual)
from itertools import combinations

class UnigramTokenizer:
    def __init__(self, vocab_size=100):
        self.vocab_size = vocab_size
        self.vocab = {}  # token -> prob

    def _get_candidates(self, word_freqs, max_len=10):
        candidates = Counter()
        for word, freq in word_freqs.items():
            for i in range(len(word)):
                for j in range(i+1, min(i+max_len+1, len(word)+1)):
                    candidates[word[i:j]] += freq
        return candidates

    def _segment(self, word, vocab):
        """Optimal segmentation using Viterbi."""
        n = len(word)
        # dp[i] = (best_log_prob, best_segmentation)
        dp = [(-float('inf'), None)] * (n + 1)
        dp[0] = (0, [])
        for i in range(1, n + 1):
            for j in range(max(0, i - 10), i):
                token = word[j:i]
                if token in vocab:
                    log_p = np.log(vocab[token])
                    if dp[j][0] + log_p > dp[i][0]:
                        dp[i] = (dp[j][0] + log_p, dp[j][1] + [token])
        return dp[n]

    def train(self, corpus):
        word_freqs = Counter()
        for text in corpus:
            for w in text.lower().split():
                word_freqs[w] += 1
        candidates = self._get_candidates(word_freqs)
        total = sum(candidates.values())
        self.vocab = {t: c / total for t, c in candidates.items()}

        # Iteratively remove tokens with smallest loss increase
        while len(self.vocab) > self.vocab_size:
            # Compute loss increase for deleting each token
            losses = {}
            for token in list(self.vocab.keys()):
                if len(token) == 1:  # Keep single characters
                    continue
                # Loss after token removal (simplified)
                losses[token] = self.vocab[token]  # Use probability as proxy for loss
            if not losses:
                break
            # Remove tokens with smallest loss
            sorted_tokens = sorted(losses.items(), key=lambda x: x[1])
            n_remove = max(1, len(self.vocab) - self.vocab_size)
            for token, _ in sorted_tokens[:n_remove]:
                del self.vocab[token]

# Small example
uni = UnigramTokenizer(vocab_size=30)
uni.train(corpus)
print(f"Unigram Vocabulary Size: {len(uni.vocab)}")
print(f"Top tokens: {sorted(uni.vocab.items(), key=lambda x: -x[1])[:10]}")

# Segmentation example
for word in ['lowest', 'newest']:
    log_p, seg = uni._segment(word, uni.vocab)
    print(f"  {word} -> {seg} (log_p={log_p:.4f})")

## 12.5 Byte-level BPE — GPT-2/3's Choice

Problem: Too many Unicode characters (11,172 Hangul syllables, emojis, etc.).
Solution: Encode all text as **UTF-8 bytes**, then apply BPE.

- Only 256 possible bytes → base vocabulary of 256
- No UNK (unknown) token needed — every string can be represented
- Easy handling of multilingual text

GPT-2 vocabulary: 256 bytes + 50,257 merges ≈ 50K tokens

In [ ]:
# Byte-level BPE demonstration
text = "Hello, café!"
bytes_seq = text.encode('utf-8')
print(f"Original: {text}")
print(f"UTF-8 bytes: {list(bytes_seq)}")
print(f"Number of bytes: {len(bytes_seq)} (character count {len(text)})")
print("\n=> All characters can be represented as bytes. UNK unnecessary.")

# Using HuggingFace tokenizers library
try:
    from tokenizers import Tokenizer
    from tokenizers.models import BPE as HFBPE
    from tokenizers.trainers import BpeTrainer
    from tokenizers.pre_tokenizers import Whitespace
except ImportError:
    print("\n[HuggingFace tokenizers package required: pip install tokenizers]")

# Simple BPE Tokenizer (HuggingFace)
try:
    from tokenizers import Tokenizer
    from tokenizers.models import BPE as HFBPE
    from tokenizers.trainers import BpeTrainer
    from tokenizers.pre_tokenizers import Whitespace

    tokenizer = Tokenizer(HFBPE(unk_token="[UNK]"))
    tokenizer.pre_tokenizer = Whitespace()
    trainer = BpeTrainer(vocab_size=200, special_tokens=["[UNK]", "[PAD]", "[BOS]", "[EOS]"])

    # Train on small corpus
    from llm_math.data import load_tiny_shakespeare
    train_text = load_tiny_shakespeare(max_chars=5000)
    tokenizer.train_from_iterator([train_text], trainer)

    print(f"\nTrained Vocabulary Size: {tokenizer.get_vocab_size()}")
    # Encoding test
    enc = tokenizer.encode("To be or not to be")
    print(f"'To be or not to be' -> {enc.tokens}")
    print(f"IDs: {enc.ids}")
except ImportError:
    print("\n[tokenizers library not available. Install with pip install tokenizers]")

## 12.6 [CPU/GPU Benchmark ④] Tokenizer Training/Encoding Speed

Tokenizer operations are performed on the CPU, but speed varies significantly depending on vocabulary size and algorithm.

In [ ]:
# Tokenizer benchmark (CPU only)
import time
from llm_math.data import load_tiny_shakespeare

# Prepare large corpus
text = load_tiny_shakespeare(max_chars=100000)
print(f"Text for benchmark: {len(text):,} chars")

# 1. Measure training time with our BPE
def bench_bpe_train(text, n_merges, repeat=3):
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        bpe = BPE(num_merges=n_merges)
        bpe.train([text])
        times.append(time.perf_counter() - t0)
    return np.mean(times), np.std(times)

print(f"\n{'merges':>8} | {'train (s)':>12}")
print("-" * 25)
for n in [10, 50, 100, 200]:
    m, s = bench_bpe_train(text, n, repeat=2)
    print(f"{n:>8} | {m:>10.4f}±{s:.4f}")

# 2. Encoding speed
bpe_small = BPE(num_merges=50)
bpe_small.train([text[:5000]])

def bench_encode(bpe, text, n_words=1000):
    words = text.split()[:n_words]
    t0 = time.perf_counter()
    for w in words:
        bpe.encode(w)
    return (time.perf_counter() - t0) * 1000

t_ms = bench_encode(bpe_small, text, n_words=1000)
print(f"\n1000 words encoding: {t_ms:.2f}ms ({t_ms/1000:.3f}ms/word)")
print("\n=> Tokenizer is CPU work. Larger vocabulary makes search slower.")
print("   HuggingFace tokenizers are Rust implementation, so much faster.")

## 12.7 Vocabulary Size and Performance Trade-Off

- Smaller vocabulary → longer sequences, slower model, smaller embedding table
- Larger vocabulary → shorter sequences, larger embedding table, insufficient learning for rare tokens

Vocabulary Sizes Across LLMs:
- GPT-2: 50,257
- GPT-3: 50,257
- LLaMA-2: 32,000
- LLaMA-3: 128,256
- GPT-4: ~100,000

In [ ]:
# Comparing sequence length by vocabulary size (simulation)
# Smaller vocabularies split into more tokens
np.random.seed(0)
text_sample = load_tiny_shakespeare(max_chars=5000)

# Train with various vocabulary sizes and compare average token counts
vocab_sizes = [30, 50, 100, 200, 500]
avg_tokens = []
for vs in vocab_sizes:
    bpe = BPE(num_merges=vs - 30)  # Initial vocabulary ~30
    bpe.train([text_sample])
    # Average token count for sample words
    sample_words = text_sample.split()[:100]
    total_tokens = sum(len(bpe.encode(w)) for w in sample_words)
    avg_tokens.append(total_tokens / len(sample_words))

plt.figure(figsize=(9, 5))
plt.plot(vocab_sizes, avg_tokens, 'o-', linewidth=2, markersize=8)
plt.xlabel('Vocabulary Size')
plt.ylabel('Mean token count / word')
plt.title('Trade-off between Vocabulary Size and Sequence Length')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch12_vocab_size_tradeoff.png', dpi=100, bbox_inches='tight')
plt.show()
print("=> As vocabulary size increases, token count per word decreases. But embedding table grows.")

## 12.8 Key Takeaways

| Algorithm | Merge Criterion | Used Model |
|---|---|---|
| BPE | Maximum frequency | GPT-2 |
| WordPiece | $\frac{\text{freq}(ab)}{\text{freq}(a)\text{freq}(b)}$ | BERT |
| Unigram | Minimum loss (top-down) | T5 |
| Byte-level BPE | BPE on bytes | GPT-2/3/4 |

## Exercises

1. Apply your own BPE implementation to Korean text and describe the challenges encountered.
2. Train BPE and WordPiece on the same corpus and compare their merge orders.
3. Compare the average number of tokens for vocabulary sizes of 100, 500, and 1000.
4. Explain why Byte-level BPE eliminates the UNK token.
5. Load the GPT-2 tokenizer using the HuggingFace `tokenizers` library and encode Korean text.

> Solution: `solutions/ch12_solutions.ipynb`